# Hierarchical Planning

## Imports

In [1]:
import os
import time
import torch
import random
import minari
import numpy as np
import pprint

from dataclasses import dataclass
from torch import nn, Tensor
from torch.utils.data import DataLoader

# flow_matching
from flow_matching.path.scheduler import CondOTScheduler
from flow_matching.path import AffineProbPath
from flow_matching.solver import ODESolver
from flow_matching.utils import ModelWrapper

# visualization
import matplotlib.pyplot as plt
from matplotlib import cm

# training and evaluation
from src.run import (
    train
)

from src.pipelines.preprocessing import (
    collate_fn,
    get_dataset_stats,
    create_trajectory_chunks,
    create_normalized_chunks,
)
from src.pipelines.eval import (
    WrappedModel,
    WrappedConditionalModel,
    evaluate_open_loop,
    evaluate_policy_mpc,
)
from src.pipelines.sampling import generate_trajectory, unnormalize_trajectory
from src.models.backbone import MLP, CNN, ConditionalCNN, ConditionalUNet1D
from src.utils.loggers import WandBLogger

# visualization and evaluation
from src.pipelines.lunarlander.visualizers import visualize_trajectories as hvt
from src.pipelines.hopper.visualizers import visualize_trajectory as lvt

# To avoid meshgrid warning
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="torch")

if torch.cuda.is_available():
    device = "cuda:0"
    print("Using gpu")
else:
    device = "cpu"
    print("Using cpu.")
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

Using cpu.


## CCNN Global Planner

In [22]:
from src.run import load_dataset
@dataclass
class LunarLanderArgs:
    environment: str = "LunarLander-v3"
    horizon: int = 100
    batch_size: int = 32
    num_epochs: int = 100
    print_every: int = 10
    lr: float = 1e-3
    hidden_dim: int = 128
    kernel_size: int = 5
    step_size: float = 0.05
    solver_method: str = "midpoint"
    inference_batch_size: int = 1
    condition_on: str = "start_obs_goal"
    model_type: str = "ccnn"
    device: str = "cuda:0" if torch.cuda.is_available() else "cpu"
    model_target: str = "obs_only"
args = LunarLanderArgs()

# Preparing the dataset
config, minari_dataset, env = load_dataset(args)
action_dim = config.action_dim
obs_dim = config.obs_dim
input_dim = obs_dim * args.horizon
cond_dim = obs_dim * 2
minari_dataset_stats = get_dataset_stats(minari_dataset)
dataloader = DataLoader(minari_dataset, batch_size=args.batch_size, shuffle=True, collate_fn=collate_fn)

SAVE_DIR = "src/checkpoints"
run_name = f"{args.environment}_{args.model_type}_h{args.horizon}_e{args.num_epochs}_k{args.kernel_size}_{args.condition_on}"
MODEL_NAME = run_name + ".pth"
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, MODEL_NAME)
pp = pprint.PrettyPrinter(indent=2)
print("Training configuration:")
pp.pprint(config)
pp.pprint(args)
print("Run name:", run_name)


Training configuration:
LunarLanderConfig(env_name='LunarLander-v3',
                  dataset_name='Box2D/LunarLanderContinuous-v3/expert-v0',
                  obs_dim=8,
                  action_dim=2,
                  category='Box2D')
LunarLanderArgs(environment='LunarLander-v3',
                horizon=100,
                batch_size=32,
                num_epochs=100,
                print_every=10,
                lr=0.001,
                hidden_dim=128,
                kernel_size=5,
                step_size=0.05,
                solver_method='midpoint',
                inference_batch_size=1,
                condition_on='start_obs_goal',
                model_type='ccnn',
                device='cpu',
                model_target='obs_only')
Run name: LunarLander-v3_ccnn_h100_e100_k5_start_obs_goal


In [ ]:
logger = WandBLogger(config=args, run_name=run_name)
model, stats, input_dim = train(
    config=config, args=args, env=env, dataset=minari_dataset, logger=logger
)
env = minari_dataset.recover_environment()
evaluate_open_loop(env, model, stats, input_dim, args, logger=logger)
logger.finish()

## UNet Global Planner
This one predicts observations only

In [18]:
from src.run import load_dataset


@dataclass
class LunarLanderArgs:
    environment: str = "LunarLander-v3"
    horizon: int = 100
    batch_size: int = 32
    num_epochs: int = 100
    print_every: int = 10
    lr: float = 1e-3
    hidden_dim: int = 64
    kernel_size: int = 5
    step_size: float = 0.05
    solver_method: str = "midpoint"
    inference_batch_size: int = 1
    condition_on: str = "start_obs_goal"
    model_type: str = "unet"
    device: str = "cuda:0" if torch.cuda.is_available() else "cpu"
    model_target: str = "obs_only"


args = LunarLanderArgs()

# Preparing the dataset
config, minari_dataset, env = load_dataset(args)
action_dim = config.action_dim
obs_dim = config.obs_dim
input_dim = obs_dim * args.horizon
cond_dim = obs_dim * 2
minari_dataset_stats = get_dataset_stats(minari_dataset)
dataloader = DataLoader(
    minari_dataset, batch_size=args.batch_size, shuffle=True, collate_fn=collate_fn
)

SAVE_DIR = "src/checkpoints"
run_name = f"{args.environment}_{args.model_type}_h{args.horizon}_e{args.num_epochs}_k{args.kernel_size}_{args.condition_on}_GLOBAL"
MODEL_NAME = run_name + ".pth"
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, MODEL_NAME)
pp = pprint.PrettyPrinter(indent=2)
print("Training configuration:")
pp.pprint(config)
pp.pprint(args)
print("Run name:", run_name)

Training configuration:
LunarLanderConfig(env_name='LunarLander-v3',
                  dataset_name='Box2D/LunarLanderContinuous-v3/expert-v0',
                  obs_dim=8,
                  action_dim=2,
                  category='Box2D')
LunarLanderArgs(environment='LunarLander-v3',
                horizon=100,
                batch_size=32,
                num_epochs=100,
                print_every=10,
                lr=0.001,
                hidden_dim=64,
                kernel_size=5,
                step_size=0.05,
                solver_method='midpoint',
                inference_batch_size=1,
                condition_on='start_obs_goal',
                model_type='unet',
                device='cpu',
                model_target='obs_only')
Run name: LunarLander-v3_unet_h100_e100_k5_start_obs_goal_GLOBAL


In [ ]:
logger = WandBLogger(
    config=args,
    run_name=run_name
)
model, stats, input_dim = train( 
    config=config, args=args, dataset=minari_dataset, logger=logger
)
env = minari_dataset.recover_environment()
evaluate_open_loop(env, model, stats, input_dim, args, logger=logger)
logger.finish()

## UNet Local Planner
This one predicts actions given observations generated by the global planner.

In [19]:
from src.run import load_dataset, build_dataloader
@dataclass
class LunarLanderArgs:
    environment: str = "LunarLander-v3"
    horizon: int = 100
    batch_size: int = 32
    num_epochs: int = 100
    print_every: int = 10
    lr: float = 1e-3
    hidden_dim: int = 64
    kernel_size: int = 5
    step_size: float = 0.05
    solver_method: str = "midpoint"
    inference_batch_size: int = 1
    condition_on: str = "start_obs_goal"
    model_type: str = "unet"
    device: str = "cuda:0" if torch.cuda.is_available() else "cpu"
    model_target: str = "act_only"
args = LunarLanderArgs()

# Preparing the dataset
config, minari_dataset, env = load_dataset(args)
action_dim = config.action_dim
obs_dim = config.obs_dim
input_dim = action_dim * args.horizon
cond_dim = obs_dim * 2
minari_dataset_stats = get_dataset_stats(minari_dataset)
dataloader = DataLoader(minari_dataset, batch_size=args.batch_size, shuffle=True, collate_fn=collate_fn)

SAVE_DIR = "src/checkpoints"
run_name = f"{args.environment}_{args.model_type}_h{args.horizon}_e{args.num_epochs}_k{args.kernel_size}_{args.condition_on}_LOCAL"
MODEL_NAME = run_name + ".pth"
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, MODEL_NAME)
pp = pprint.PrettyPrinter(indent=2)
print("Training configuration:")
pp.pprint(config)
pp.pprint(args)
print("Run name:", run_name)

Training configuration:
LunarLanderConfig(env_name='LunarLander-v3',
                  dataset_name='Box2D/LunarLanderContinuous-v3/expert-v0',
                  obs_dim=8,
                  action_dim=2,
                  category='Box2D')
LunarLanderArgs(environment='LunarLander-v3',
                horizon=100,
                batch_size=32,
                num_epochs=100,
                print_every=10,
                lr=0.001,
                hidden_dim=64,
                kernel_size=5,
                step_size=0.05,
                solver_method='midpoint',
                inference_batch_size=1,
                condition_on='start_obs_goal',
                model_type='unet',
                device='cpu',
                model_target='act_only')
Run name: LunarLander-v3_unet_h100_e100_k5_start_obs_goal_LOCAL


In [ ]:
logger = WandBLogger(
    config=args,
    run_name=run_name
)
model, stats, input_dim = train( 
    config=config, args=args, dataset=minari_dataset, logger=logger
)
logger.finish()

## MPC Evaluation
1. call the global planner to get a sequence of observations
2. use the output of the global planner as conditioning signals for the local planner

In [20]:
MODEL_SAVE_PATH = "src/checkpoints/LunarLander-v3_unet_h25_e100_k5_start_obs_goal_LOCAL.pth"
print(f"Loading model from {MODEL_SAVE_PATH}...")
state_dict = torch.load(MODEL_SAVE_PATH, map_location=device)
vf = ConditionalUNet1D(
    horizon=args.horizon,
    input_dim=input_dim,
    hidden_dim=args.hidden_dim,
    cond_dim=cond_dim,
).to(device)
vf.load_state_dict(state_dict)
wrapped_vf = WrappedConditionalModel(vf)
step_size = 0.05
T = torch.linspace(0, 1, 10)  # sample times
T = T.to(device=device)
solver = ODESolver(velocity_model=wrapped_vf)  # create an ODESolver class

Loading model from src/checkpoints/LunarLander-v3_unet_h25_e100_k5_start_obs_goal_LOCAL.pth...


### Testing local planner
We first test the local planner on its own...

In [ ]:
num_eval_episodes = 100
final_goal_state = torch.tensor([0, 0, 0, 0, 0, 0, 1, 1], dtype=torch.float32)

local_planner = lambda cond_dict: generate_trajectory(
    stats=minari_dataset_stats,
    solver=solver,
    T=T,
    input_dim=input_dim,
    horizon=args.horizon,
    condition=cond_dict,
    batch_size=args.inference_batch_size,
    model_target=args.model_target,
)
model_rewards = evaluate_policy_mpc(
    env,
    local_planner,
    num_eval_episodes,
    replan_freq=5,
    render=False,
    max_episode_length=100,
    condition_type="start_obs_goal",
    goal_obs= final_goal_state,
)
env.close()
avg_model_reward = np.mean(model_rewards)
std_model_reward = np.std(model_rewards)
print(
    f"Average MPC Reward over {num_eval_episodes} episodes: {avg_model_reward:.2f} +/- {std_model_reward:.2f}"
)


--- Starting MPC Evaluation (Condition Type: start_obs_goal) and Replan Frequency: 5 ---
Episode 1/10 finished. Total Reward: 67.16
Episode 2/10 finished. Total Reward: 113.29
Episode 3/10 finished. Total Reward: 141.37
Episode 4/10 finished. Total Reward: 71.60
Episode 5/10 finished. Total Reward: 65.42
Episode 6/10 finished. Total Reward: 146.72
Episode 7/10 finished. Total Reward: 163.96
Episode 8/10 finished. Total Reward: 135.84
Episode 9/10 finished. Total Reward: 44.77
Episode 10/10 finished. Total Reward: 15.82
Average MPC Reward over 10 episodes: 96.60 +/- 47.49


In [17]:
GLOBAL_MODEL_PATH = "src/checkpoints/LunarLander-v3_unet_h100_e100_k5_start_obs_goal.pth"
global_args = LunarLanderArgs(model_target="obs_only")            # horizon, hidden_dim…
obs_input_dim = config.obs_dim * global_args.horizon
gmodel = ConditionalUNet1D(
    horizon=global_args.horizon, input_dim=obs_input_dim,
    hidden_dim=global_args.hidden_dim, cond_dim=config.obs_dim * 2
).to(device)
gmodel.load_state_dict(torch.load(GLOBAL_MODEL_PATH, map_location=device))
gsolver = ODESolver(WrappedConditionalModel(gmodel))
gT = torch.linspace(0, 1, 10).to(device)

# Local model (act | start_obs, end_obs)
LOCAL_MODEL_PATH = "src/checkpoints/LunarLander-v3_unet_h25_e100_k5_start_obs_goal_LOCAL.pth"
local_args = LunarLanderArgs(model_target="act_only")             # horizon, hidden_dim…
act_input_dim = config.action_dim * local_args.horizon
lmodel = ConditionalUNet1D(
    horizon=local_args.horizon, input_dim=act_input_dim,
    hidden_dim=local_args.hidden_dim, cond_dim=config.obs_dim * 2
).to(device)
lmodel.load_state_dict(torch.load(LOCAL_MODEL_PATH, map_location=device))
lsolver = ODESolver(WrappedConditionalModel(lmodel))
lT = torch.linspace(0, 1, 10).to(device)

# Wrappers that sample from each planner
global_planner = lambda cond: generate_trajectory(
    stats=minari_dataset_stats, solver=gsolver, T=gT,
    input_dim=obs_input_dim, horizon=global_args.horizon,
    condition=cond, batch_size=1, model_target="obs_only"
)
local_planner = lambda cond: generate_trajectory(
    stats=minari_dataset_stats, solver=lsolver, T=lT,
    input_dim=act_input_dim, horizon=local_args.horizon,
    condition=cond, batch_size=1, model_target="act_only"
)

# ---------- Hierarchical MPC evaluation --------------------------------------
replan_freq = 5                         # how often to replan
final_goal_state = torch.tensor([0,0,0,0,0,0,1,1], dtype=torch.float32)

def hierarchical_planner(cond_dict):
    # 1) High‑level plan in observation space
    obs_plan, _ = global_planner(cond_dict)

    # choose waypoint for the low‑level controller
    waypoint_idx = min(replan_freq, len(obs_plan)-1)
    subgoal = obs_plan[waypoint_idx]

    # 2) Low‑level plan in action space conditioned on the waypoint
    start_obs = cond_dict["start_obs_goal"][0]
    local_cond = {"start_obs_goal": (start_obs, subgoal)}
    _, act_plan = local_planner(local_cond)

    return obs_plan, act_plan

env = minari_dataset.recover_environment(eval_env=False, render_mode="rgb_array")
num_eval_episodes = 100

rewards = evaluate_policy_mpc(
    env,
    hierarchical_planner,
    num_eval_episodes,
    condition_type="start_obs_goal",
    goal_obs=final_goal_state,
    max_episode_length=300,
    replan_freq=replan_freq,
    render=False,
)
print(f"Average reward: {np.mean(rewards):.2f} ± {np.std(rewards):.2f}")
env.close()


--- Starting MPC Evaluation (Condition Type: start_obs_goal) and Replan Frequency: 5 ---
Episode 1/100 finished. Total Reward: 31.31
Episode 2/100 finished. Total Reward: 14.77
Episode 3/100 finished. Total Reward: 8.93
Episode 4/100 finished. Total Reward: 68.58
Episode 5/100 finished. Total Reward: 154.79
Episode 6/100 finished. Total Reward: 43.25
Episode 7/100 finished. Total Reward: 21.54
Episode 8/100 finished. Total Reward: 26.60
Episode 9/100 finished. Total Reward: 183.20
Episode 10/100 finished. Total Reward: 176.34
Episode 11/100 finished. Total Reward: 56.72
Episode 12/100 finished. Total Reward: 184.52
Episode 13/100 finished. Total Reward: 175.20
Episode 14/100 finished. Total Reward: 176.09
Episode 15/100 finished. Total Reward: 165.59
Episode 16/100 finished. Total Reward: 54.26
Episode 17/100 finished. Total Reward: 137.84
Episode 18/100 finished. Total Reward: 56.47
Episode 19/100 finished. Total Reward: 145.38
Episode 20/100 finished. Total Reward: 34.31
Episode 21/